# TF-IDF baseline for recommendation status

The input is `data/tokenized/digikala-comments-tokenized.parquet`. Polars loads it. The target is `recommendation_status`. `rate` is not a feature and is not the label. The only features are the tokenizer ids.

Classes, with codes `recommended=0`, `not_recommended=1`, `no_idea=2`:

- `recommended`
- `not_recommended`
- `no_idea`

A missing recommendation is left out of this classifier. Each product is placed in one split: 70% train, 15% validation, 15% test. Reviews of one product stay together. The TF-IDF vocabulary is fit on the training split only, then a logistic regression predicts `recommendation_status`.


In [16]:
from pathlib import Path
import hashlib
import sys

import polars as pl
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import AutoTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TOKENIZED = ROOT / "data/tokenized/digikala-comments-tokenized.parquet"
TOKENIZER_NAME = "HooshvareLab/bert-fa-base-uncased"
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15
LABELS = ["recommended", "not_recommended", "no_idea"]
LABEL_CODE = {label: code for code, label in enumerate(LABELS)}
SPLITS = ["train", "validation", "test"]
SPLIT_SHARE = {"train": TRAIN_SHARE, "validation": VALIDATION_SHARE, "test": TEST_SHARE}


## Label `recommendation_status`

The three classes stay as written in the cleaned file. A null recommendation is counted, then dropped before the split.


In [17]:
reviews = pl.read_parquet(TOKENIZED).with_columns(
    pl.col("recommendation_status").replace_strict(LABEL_CODE, default=None).alias("label_id")
)

label_counts = (
    reviews.group_by("recommendation_status")
    .agg(pl.len().alias("rows"))
    .with_columns((pl.col("rows") / reviews.height * 100).round(2).alias("percent"))
    .sort("recommendation_status")
)
print("rows:", reviews.height)
print("missing recommendation:", reviews.filter(pl.col("recommendation_status").is_null()).height)
label_counts


rows: 5258351
missing recommendation: 0


recommendation_status,rows,percent
str,u32,f64
"""no_idea""",593653,11.29
"""not_recommended""",504465,9.59
"""recommended""",4160233,79.12


## Split by product

Products are ordered by a hash of `SPLIT_SEED` and `product_id`, then assigned to the split that is furthest below its target share of each recommendation class. A product's reviews all receive that one split. Rows with a missing recommendation are not used to train or score this classifier.


In [18]:
def product_order_key(product_id: int) -> int:
    key = f"{SPLIT_SEED}|{product_id}".encode()
    return int(hashlib.sha256(key).hexdigest()[:16], 16)


labeled = reviews.filter(pl.col("recommendation_status").is_in(LABELS))
products = (
    labeled.group_by("product_id")
    .agg(
        pl.len().alias("n"),
        *[(pl.col("recommendation_status") == label).sum().alias(f"n_{label}") for label in LABELS],
    )
    .sort("product_id")
)
ordered = sorted(
    products.iter_rows(named=True),
    key=lambda row: (product_order_key(row["product_id"]), row["product_id"]),
)
totals = {label: sum(row[f"n_{label}"] for row in ordered) for label in LABELS}
targets = {
    split: {label: SPLIT_SHARE[split] * totals[label] for label in LABELS}
    for split in SPLITS
}
filled = {split: {label: 0 for label in LABELS} for split in SPLITS}


def split_room(split: str, row: dict) -> float:
    score = 0.0
    for label in LABELS:
        target = targets[split][label]
        if target == 0:
            continue
        score += row[f"n_{label}"] * (target - filled[split][label]) / target
    return score


assignments = {}
for row in ordered:
    chosen = max(SPLITS, key=lambda split: split_room(split, row))
    assignments[row["product_id"]] = chosen
    for label in LABELS:
        filled[chosen][label] += row[f"n_{label}"]

split_frame = pl.DataFrame(
    {"product_id": list(assignments), "split": list(assignments.values())}
)
reviews = reviews.join(split_frame, on="product_id", how="left")
labeled = reviews.filter(pl.col("recommendation_status").is_in(LABELS))

shared_products = (
    labeled.group_by("product_id")
    .agg(pl.col("split").n_unique().alias("n_splits"))
    .filter(pl.col("n_splits") > 1)
    .height
)
print("products in more than one split:", shared_products)

split_counts = (
    labeled.group_by("split", "recommendation_status")
    .agg(pl.len().alias("rows"))
    .with_columns((pl.col("rows") / labeled.height * 100).round(2).alias("percent_of_labeled"))
    .sort("split", "recommendation_status")
)
split_counts


products in more than one split: 0


split,recommendation_status,rows,percent_of_labeled
str,str,u32,f64
"""test""","""no_idea""",89061,1.69
"""test""","""not_recommended""",75677,1.44
"""test""","""recommended""",624030,11.87
"""train""","""no_idea""",415546,7.9
"""train""","""not_recommended""",353117,6.72
"""train""","""recommended""",2912159,55.38
"""validation""","""no_idea""",89046,1.69
"""validation""","""not_recommended""",75671,1.44
"""validation""","""recommended""",624044,11.87


## TF-IDF and logistic regression

Each review is the space-separated tokenizer ids from `input_ids`, without the classifier, separator, or pad ids. `TfidfVectorizer` sees the training documents only. Logistic regression then predicts `recommended`, `not_recommended`, and `no_idea`. Class weights are balanced from the training rows. `rate` is not passed to the vectorizer or the classifier.


In [19]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
special_ids = [
    token_id
    for token_id in (tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id)
    if token_id is not None
]

documents = labeled.with_columns(
    pl.col("input_ids")
    .list.eval(
        pl.when(pl.element().is_in(special_ids))
        .then(None)
        .otherwise(pl.element().cast(pl.String))
    )
    .list.drop_nulls()
    .list.join(" ")
    .alias("token_text")
).select("id", "product_id", "split", "recommendation_status", "label_id", "token_text")

train = documents.filter(pl.col("split") == "train")
validation = documents.filter(pl.col("split") == "validation")
test = documents.filter(pl.col("split") == "test")
print("train", train.height, "validation", validation.height, "test", test.height)

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\d+\b", min_df=5)
x_train = vectorizer.fit_transform(train.get_column("token_text").to_list())
y_train = train.get_column("label_id").to_numpy()
print("vocabulary:", len(vectorizer.vocabulary_))

classifier = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=200,
    random_state=SPLIT_SEED,
)
classifier.fit(x_train, y_train)


def score_split(name: str, frame: pl.DataFrame) -> None:
    features = vectorizer.transform(frame.get_column("token_text").to_list())
    actual = frame.get_column("label_id").to_numpy()
    predicted = classifier.predict(features)
    print(name)
    print("accuracy:", round(float(accuracy_score(actual, predicted)), 4))
    print("balanced accuracy:", round(float(balanced_accuracy_score(actual, predicted)), 4))
    print("macro f1:", round(float(f1_score(actual, predicted, average="macro")), 4))
    print(classification_report(actual, predicted, labels=[0, 1, 2], target_names=LABELS, digits=4))
    print(confusion_matrix(actual, predicted, labels=[0, 1, 2]))
    print()


score_split("validation", validation)
score_split("test", test)


train 3680822 validation 788761 test 788768
vocabulary: 32008


/Users/amir/miniconda3/envs/maktab/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


validation
accuracy: 0.758
balanced accuracy: 0.6892
macro f1: 0.6116
                 precision    recall  f1-score   support

    recommended     0.9595    0.7925    0.8681    624044
not_recommended     0.4839    0.7664    0.5932     75671
        no_idea     0.2952    0.5088    0.3736     89046

       accuracy                         0.7580    788761
      macro avg     0.5795    0.6892    0.6116    788761
   weighted avg     0.8389    0.7580    0.7859    788761

[[494576  35759  93709]
 [  3233  57996  14442]
 [ 17648  26095  45303]]

test
accuracy: 0.7569
balanced accuracy: 0.6889
macro f1: 0.6108
                 precision    recall  f1-score   support

    recommended     0.9601    0.7911    0.8674    624030
not_recommended     0.4828    0.7667    0.5925     75677
        no_idea     0.2936    0.5089    0.3724     89061

       accuracy                         0.7569    788768
      macro avg     0.5788    0.6889    0.6108    788768
   weighted avg     0.8390    0.7569    0.785

## Save the fitted baseline

The vectorizer and the classifier go in one file under `models/`. That directory stays out of git. Only `models/.gitkeep` is tracked.


In [20]:
import joblib

model_dir = ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "recommendation-tfidf-logreg.joblib"
joblib.dump(
    {
        "vectorizer": vectorizer,
        "classifier": classifier,
        "labels": LABELS,
        "label_code": LABEL_CODE,
        "tokenizer_name": TOKENIZER_NAME,
        "split_seed": SPLIT_SEED,
    },
    model_path,
)
print(model_path)
print("bytes:", model_path.stat().st_size)


/Users/amir/Documents/GitHub/digikala-products-recommender-systems/models/recommendation-tfidf-logreg.joblib
bytes: 1371227
